# 02 - Model Training

Load the folder-based FER-2013 images with Keras and train the MobileNetV2 transfer learning model.


In [1]:
import sys
from pathlib import Path

print('Notebook Python:', sys.executable)
current = Path.cwd().resolve()
expected_python = None
for path in [current, *current.parents]:
    candidate = path / '.venv' / 'bin' / 'python'
    if candidate.exists():
        expected_python = candidate
        break

if expected_python and Path(sys.executable).resolve() != expected_python.resolve():
    print('WARNING: This notebook is not using the project .venv kernel.')
    print('Expected:', expected_python)
    print('In VS Code, select kernel: KidMood (.venv)')
else:
    print('Kernel check passed.')


Notebook Python: /home/daddy/Desktop/_DEV/_SCHOOL/ITAI-1378_COMP-VISION/KidMood-Final-Project/.venv/bin/python
Kernel check passed.


In [2]:
from pathlib import Path
import os
import sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / 'src' / 'data_processing.py').exists():
            return path
    raise FileNotFoundError('Could not find project root containing src/data_processing.py')

PROJECT_ROOT = find_project_root()
src_path = PROJECT_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import matplotlib.pyplot as plt
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print('Memory growth was not set:', exc)

print("TensorFlow:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices("GPU"))

from data_processing import build_image_datasets, optimize_dataset, write_dataset_summary
from model import build_mobilenet_model

raw_dir = PROJECT_ROOT / 'data' / 'raw'
processed_dir = PROJECT_ROOT / 'data' / 'processed'
write_dataset_summary(raw_dir, processed_dir)
print('Project root:', PROJECT_ROOT)
print('Dataset metadata saved to:', processed_dir)
print('GPUs:', tf.config.list_physical_devices('GPU'))


I0000 00:00:1778279403.993181   28246 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778279404.019789   28246 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778279404.615746   28246 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0
Built with CUDA: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Project root: /home/daddy/Desktop/_DEV/_SCHOOL/ITAI-1378_COMP-VISION/KidMood-Final-Project
Dataset metadata saved to: /home/daddy/Desktop/_DEV/_SCHOOL/ITAI-1378_COMP-VISION/KidMood-Final-Project/data/processed
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
train_ds, val_ds, _ = build_image_datasets(raw_dir=raw_dir, batch_size=32)
train_ds = optimize_dataset(train_ds)
val_ds = optimize_dataset(val_ds)


Found 20137 files belonging to 4 classes.
Using 17117 files for training.


I0000 00:00:1778279413.196876   28246 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1080 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9


Found 20137 files belonging to 4 classes.
Using 3020 files for validation.
Found 5003 files belonging to 4 classes.


In [4]:
model = build_mobilenet_model(num_classes=4)
model.summary()


Model: "KidMood_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [5]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
)

model_path = PROJECT_ROOT / 'models' / 'trained' / 'kidmood_mobilenetv2.keras'
model_path.parent.mkdir(parents=True, exist_ok=True)
model.save(model_path)
print('Model saved to:', model_path)


Epoch 1/5


I0000 00:00:1778279422.055451   33630 service.cc:153] XLA service 0x718b0c05e450 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778279422.055521   33630 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.21.1)
I0000 00:00:1778279422.120708   33630 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778279422.456840   33630 cuda_dnn.cc:461] Loaded cuDNN version 92101
I0000 00:00:1778279422.488211   33630 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8072__.119
E0000 00:00:1778279423.699732   33630 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1778279425.927530   33630 dot_search_space.cc:240] All configs w

528/535 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3988 - loss: 1.3310

I0000 00:00:1778279435.004314   33634 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8072__.119
E0000 00:00:1778279436.763176   33634 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


535/535 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3993 - loss: 1.3298

E0000 00:00:1778279445.159881   33630 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


535/535 ━━━━━━━━━━━━━━━━━━━━ 30s 39ms/step - accuracy: 0.4372 - loss: 1.2425 - val_accuracy: 0.5106 - val_loss: 1.1253
Epoch 2/5
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.4993 - loss: 1.1327 - val_accuracy: 0.5242 - val_loss: 1.0893
Epoch 3/5
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5239 - loss: 1.0879 - val_accuracy: 0.5450 - val_loss: 1.0539
Epoch 4/5
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5371 - loss: 1.0616 - val_accuracy: 0.5550 - val_loss: 1.0287
Epoch 5/5
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5426 - loss: 1.0523 - val_accuracy: 0.5560 - val_loss: 1.0301
Model saved to: /home/daddy/Desktop/_DEV/_SCHOOL/ITAI-1378_COMP-VISION/KidMood-Final-Project/models/trained/kidmood_mobilenetv2.keras
